# FedUCA Centralised Baseline

Trains Bottleneck N/4 on ALL training data (no federation).
This is the upper bound — the best any FL method could hope to achieve.

**Add data:** `ham10000-vit-embeddings-orig`

**Accelerator: None (CPU)** — runs in ~10 minutes.

In [ ]:
import os, json, glob, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RESULTS_DIR = "/kaggle/working/results_centralised"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = torch.device("cpu")
NUM_CLASSES = 7
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
SEEDS       = [0, 1, 2, 3, 4]

hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("Add ham10000-vit-embeddings-orig dataset.")
EMBED_DIR = os.path.dirname(hits[0])
print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    return (torch.tensor(d['embeddings'], dtype=torch.float32),
            torch.tensor(d['labels'],     dtype=torch.long))

emb_train, lbl_train = load_split("train")
emb_val,   lbl_val   = load_split("val")
emb_test,  lbl_test  = load_split("test")

print(f"Train: {emb_train.shape}  Val: {emb_val.shape}  Test: {emb_test.shape}")

counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = (1.0 / counts)
weights = (weights / weights.mean()).to(DEVICE)
print(f"Class weights: {weights.numpy().round(3)}")
print(f"\nClass distribution (train):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {c}: {int(counts[i])} ({counts[i]/counts.sum()*100:.1f}%)")

In [ ]:
# Bottleneck N/4 head — identical to federated experiments
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768, 4*b), nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, b),   nn.GELU(), nn.LayerNorm(b),   nn.Dropout(0.2),
            nn.Linear(b, 4*b),   nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, 7))
    def forward(self, x): return self.net(x)

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval()
    preds, targets = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        preds.append(head(x.to(DEVICE)).argmax(1).cpu())
        targets.append(y)
    p = torch.cat(preds).numpy()
    t = torch.cat(targets).numpy()
    f1s = []
    for c in range(NUM_CLASSES):
        tp = ((p==c)&(t==c)).sum()
        fp = ((p==c)&(t!=c)).sum()
        fn = ((p!=c)&(t==c)).sum()
        pr = tp / (tp + fp + 1e-9)
        rc = tp / (tp + fn + 1e-9)
        f1s.append(float(2*pr*rc/(pr+rc+1e-9)))
    return {
        "accuracy"     : float((p==t).mean()),
        "macro_f1"     : float(np.mean(f1s)),
        "f1_per_class" : [float(x) for x in f1s]
    }

print("Model and evaluation ready.")

In [ ]:
# Train centralised model — 5 seeds, same hyperparameters as federated
print("="*55)
print("CENTRALISED BASELINE — Bottleneck N/4")
print("Same LR, epochs, class weighting as federated runs")
print("="*55)

N_EPOCHS = 100    # more epochs — no communication overhead
LR       = 5e-3
BS       = 64     # larger batch — full dataset available

results = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    head  = BottleneckMLP(ratio=4).to(DEVICE)
    opt   = optim.Adam(head.parameters(), lr=LR)
    crit  = nn.CrossEntropyLoss(weight=weights)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)

    loader = DataLoader(
        TensorDataset(emb_train.to(DEVICE), lbl_train.to(DEVICE)),
        batch_size=BS, shuffle=True)

    best_val = -1; best_state = None; best_epoch = 0
    t0 = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        head.train()
        for x, y in loader:
            opt.zero_grad()
            crit(head(x), y).backward()
            opt.step()
        sched.step()

        vm = evaluate(head, emb_val.to(DEVICE), lbl_val)
        if vm['macro_f1'] > best_val:
            best_val   = vm['macro_f1']
            best_epoch = epoch
            best_state = copy.deepcopy(head.state_dict())

        if epoch % 20 == 0:
            print(f"  seed={seed} epoch={epoch:3d} "
                  f"val={vm['macro_f1']:.4f} best={best_val:.4f} "
                  f"| {time.time()-t0:.0f}s")

    # Evaluate on test set with best checkpoint
    head.load_state_dict(best_state)
    tm = evaluate(head, emb_test.to(DEVICE), lbl_test)

    res = {
        "seed"        : seed,
        "best_epoch"  : best_epoch,
        "best_val_f1" : best_val,
        "test"        : tm
    }
    results.append(res)
    print(f"  seed={seed} DONE: "
          f"test_f1={tm['macro_f1']:.4f} "
          f"test_acc={tm['accuracy']:.4f} "
          f"best_epoch={best_epoch}")

with open(f"{RESULTS_DIR}/centralised_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nResults saved.")

In [ ]:
# Final summary — exactly the numbers needed for the paper
print("\n" + "="*60)
print("CENTRALISED BASELINE — FINAL RESULTS")
print("="*60)

f1s  = [r['test']['macro_f1'] for r in results]
accs = [r['test']['accuracy']  for r in results]
epcs = [r['best_epoch']        for r in results]

print(f"\nMacro-F1 : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
print(f"Accuracy  : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Best epoch: {np.mean(epcs):.1f} ± {np.std(epcs):.1f}")

print(f"\nPer-class F1 (mean over 5 seeds):")
pc = np.array([r['test']['f1_per_class'] for r in results])
for i, c in enumerate(CLASS_NAMES):
    print(f"  {c.upper():<8}: {pc[:,i].mean():.3f} ± {pc[:,i].std():.3f}")

# Comparison table — exactly what goes in the paper
print("\n" + "="*60)
print("PAPER TABLE: Centralised vs Federated")
print("="*60)

central_f1   = np.mean(f1s)
fedavg_iid   = 0.643   # from Phase 2, alpha=1000
pga_iid      = 0.639   # from Phase 3, alpha=1000
fedavg_noniid = 0.409  # from Phase 2, alpha=0.1 (10 seeds)
pga_noniid   = 0.483   # from Phase 3, alpha=0.1 (10 seeds)

print(f"\n{'Method':<40} {'Macro-F1':>10}")
print("-"*52)
print(f"{'Centralised (no FL, upper bound)':<40} "
      f"{central_f1:.3f}±{np.std(f1s):.3f}")
print(f"{'FedAvg K=4, α=1000 (IID FL)':<40} 0.643±0.010")
print(f"{'PGA K=4,    α=1000 (IID FL)':<40} 0.639±0.013")
print(f"{'FedAvg K=4, α=0.1  (non-IID FL)':<40} 0.409±0.117")
print(f"{'PGA K=4,    α=0.1  (non-IID FL)':<40} 0.483±0.090")

print(f"\nKey ratios:")
print(f"  FedAvg/Centralised (IID): "
      f"{fedavg_iid/central_f1*100:.1f}% of centralised performance")
print(f"  PGA/Centralised (IID)   : "
      f"{pga_iid/central_f1*100:.1f}% of centralised performance")
het_loss = central_f1 - fedavg_noniid
pga_gain = pga_noniid - fedavg_noniid
print(f"  Heterogeneity loss (FedAvg): {het_loss:.3f} F1 points")
print(f"  PGA recovery              : {pga_gain:.3f} F1 points")
print(f"  PGA recovers              : {pga_gain/het_loss*100:.1f}% of loss")

print(f"\nAbstract sentence:")
print(f'  "Under near-IID conditions, FedAvg achieves '
      f'{fedavg_iid/central_f1*100:.1f}% of centralised '
      f'performance ({fedavg_iid:.3f} vs {central_f1:.3f} macro-F1). '
      f'Under strong label heterogeneity (alpha=0.1), FedAvg drops to '
      f'{fedavg_noniid:.3f} macro-F1 ({fedavg_noniid/central_f1*100:.1f}% '
      f'of centralised); PGA recovers {pga_gain/het_loss*100:.1f}% of '
      f'this loss, achieving {pga_noniid:.3f} macro-F1 '
      f'(Delta=+{pga_gain:.3f}, p=0.001)."')

print(f"\nSave output as dataset: feduca-centralised-results")